In [9]:
import numpy as np
import json
import pandas as pd
from datetime import datetime
import threading
from collections import defaultdict
from functools import wraps
from contextlib import contextmanager

# ==========================================
# 装饰器：性能监控
# ==========================================
def timing_decorator(func):
    """
    装饰器：记录函数执行时间

    使用示例:
        @timing_decorator
        def my_function():
            time.sleep(1)
            return "done"

        result = my_function()
        print(my_function.last_execution_time)  # 输出: 1.0
    """
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start

        # 存储执行时间到函数属性
        wrapper.last_execution_time = elapsed

        # 如果是类方法，也存储到实例
        if args and hasattr(args[0], '_timings'):
            if not hasattr(args[0], '_timings'):
                args[0]._timings = {}  # type: ignore
            args[0]._timings[func.__name__] = elapsed  # type: ignore

        return result
    return wrapper

@contextmanager
def timing_context(func_name="unknown"):
    """
    上下文管理器：用于代码块计时

    使用示例:
        with timing_context("attack_generation"):
            # 执行攻击生成代码
            pass
        # 会自动打印耗时
    """
    start = time.time()
    try:
        yield
    finally:
        elapsed = time.time() - start
        print(f"⏱️ {func_name} 耗时: {elapsed:.3f}秒")
        if not hasattr(timing_context, 'timings'):
            timing_context.timings = {}
        timing_context.timings[func_name] = elapsed


def get_all_timings():
    """获取所有计时记录"""
    timings = {}
    if hasattr(timing_context, 'timings'):
        timings.update(timing_context.timings)
    return timings

def print_timing_summary():
    """打印计时汇总"""
    timings = get_all_timings()
    if not timings:
        print("⏱️ 暂无计时记录")
        return

    print("\n" + "=" * 60)
    print("⏱️ 性能计时汇总")
    print("=" * 60)
    total_time = 0
    for func_name, elapsed in timings.items():
        print(f"  {func_name}: {elapsed:.3f}秒")
        total_time += elapsed
    print("-" * 60)
    print(f"  总耗时: {total_time:.3f}秒")

    if total_time > 300:
        print(f"  ⚠️ 总耗时 {total_time:.1f}秒 超过5分钟限制！")
    else:
        print(f"  ✅ 总耗时 {total_time:.1f}秒，符合≤300秒要求")
    print("=" * 60)

@timing_decorator
def run_full_test_pipeline(clean_acc, attack_results, source_acc=None, target_acc=None):
    """
    运行完整的测试流程（带计时）
    """
    print("🚀 开始完整测试流程...")

    # 1. 攻击指标计算
    with timing_context("attack_metrics"):
        attack_metrics = batch_compute_attack_metrics(clean_acc, attack_results)

    # 2. 迁移指标计算（如果有）
    migration_metrics = None
    if source_acc is not None and target_acc is not None:
        with timing_context("migration_metrics"):
            migration_metrics = compute_migration_metrics(source_acc, target_acc)

    # 3. 汇总结果
    result = {
        'attack_metrics': attack_metrics,
        'migration_metrics': migration_metrics,
        'timings': get_all_timings(),
        'total_time': sum(get_all_timings().values())
    }

    # 打印计时汇总
    print_timing_summary()

    return result
# ==========================================
# 配置部分
# ==========================================
METRICS_CONFIG = {
    'thresholds': {
        'performance_retention': 0.9,        # 性能保持率 ≥ 90%
        'migration_retention': 0.9,          # 迁移保持率 ≥ 90%
        'migration_stability': 0.05,         # 迁移稳定性 ≤ 5%
        'worst_case_accuracy': 0.7,          # 最差性能 ≥ 70%
        'clean_accuracy': 0.85,              # 干净准确率 ≥ 85%
        'attack_accuracy': 0.7,              # 攻击准确率 ≥ 70%
        'performance_degradation': 0.3,      # 性能退化 ≤ 30%
        'worst_migration_acc': 0.85          # 最差迁移性能 ≥ 85%
    },
    'requirements': {
        'migration': ['migration_retention', 'migration_stability', 'worst_migration_acc'],
        'robustness': ['performance_retention', 'performance_degradation', 'worst_case_accuracy']
    }
}

def load_metrics_config(config_path=None):
    """加载指标配置文件"""
    if config_path:
        with open(config_path, 'r') as f:
            return json.load(f)
    return METRICS_CONFIG

# ==========================================
# 使用装饰器的函数
# ==========================================
@timing_decorator
def compute_accuracy(origin_pred, adv_pred):
    """计算攻击成功率"""
    origin_pred = np.array(origin_pred, dtype=np.int64)
    adv_pred = np.array(adv_pred, dtype=np.int64)
    diff = origin_pred != adv_pred
    flip_count = diff.sum()
    total = len(origin_pred)
    success_rate = float(flip_count) / float(total) if total > 0 else 0.0
    return success_rate, int(flip_count), int(total)

@timing_decorator
def sample_data(images, labels, ratio=0.1):
    """数据采样"""
    num_samples = len(images)
    sample_count = int(num_samples * ratio)
    indices = np.random.choice(num_samples, sample_count, replace=False)
    sampled_images = images[indices]
    sampled_labels = labels[indices]
    return sampled_images, sampled_labels

@timing_decorator
def compute_all_metrics(clean_acc, attack_acc, all_attack_accs=None):
    """计算所有评价指标（攻击鲁棒性）"""
    retention = attack_acc / clean_acc if clean_acc > 0 else 0.0
    degradation = 1.0 - retention
    worst_case = float(min(all_attack_accs)) if all_attack_accs else float(attack_acc)

    return {
        "clean_accuracy": float(clean_acc),
        "attack_accuracy": float(attack_acc),
        "performance_retention": float(retention),
        "performance_degradation": float(degradation),
        "worst_case_accuracy": float(worst_case),
    }

@timing_decorator
def compute_migration_metrics(source_acc, target_acc, history_accs=None):
    """计算迁移学习评估指标"""
    retention = target_acc / source_acc if source_acc > 0 else 0.0
    stability = float(np.std(history_accs)) if history_accs and len(history_accs) > 1 else 0.0
    worst = float(min(history_accs)) if history_accs else float(target_acc)
    failed = retention < 0.9

    return {
        "migration_retention": float(retention),
        "migration_stability": float(stability),
        "worst_migration_acc": float(worst),
        "migration_failed": bool(failed),
    }

@timing_decorator
def batch_compute_attack_metrics(clean_acc, attack_results):
    """批量计算多个攻击强度下的鲁棒性指标"""
    results = {}
    all_accs = []

    for result in attack_results:
        attack_name = result['attack_name']
        attack_acc = result['attack_acc']
        all_accs.append(float(attack_acc))

        metrics = compute_all_metrics(clean_acc, attack_acc, all_accs)
        results[attack_name] = {
            'attack_accuracy': float(attack_acc),
            'performance_retention': metrics['performance_retention'],
            'performance_degradation': metrics['performance_degradation'],
            'params': result.get('params', {})
        }

    results['summary'] = {
        'worst_case_accuracy': float(min(all_accs)),
        'avg_attack_accuracy': float(np.mean(all_accs)),
        'num_attacks': len(attack_results)
    }

    return results

@timing_decorator
def compute_comprehensive_robustness_metrics(clean_acc, attack_accs_by_type):
    """计算全面的鲁棒性指标"""
    results = {}

    for attack_type, accs in attack_accs_by_type.items():
        attack_acc_list = [float(v) for v in accs.values()]
        all_attack_accs = attack_acc_list

        for eps, acc in accs.items():
            metrics = compute_all_metrics(clean_acc, float(acc), all_attack_accs)
            results[f"{attack_type}_{eps}"] = metrics

    all_accs = []
    for attack_type, accs in attack_accs_by_type.items():
        all_accs.extend([float(v) for v in accs.values()])

    results['global_summary'] = {
        'global_worst_case': float(min(all_accs)),
        'global_avg_case': float(np.mean(all_accs)),
        'max_degradation': float(max([1 - acc/clean_acc for acc in all_accs])),
        'num_attacks_tested': len(all_accs)
    }

    return results

# ==========================================
# 第一部分：compute_accuracy（修复兼容列表/数组）
# ==========================================
def compute_accuracy(origin_pred, adv_pred):
    # 统一转为numpy数组，兼容list、numpy数组输入
    origin_pred = np.array(origin_pred)
    adv_pred = np.array(adv_pred)
    # 对比标签是否发生变化
    diff = origin_pred != adv_pred
    flip_count = diff.sum()
    total = len(origin_pred)
    success_rate = flip_count / total
    # 返回三元组：攻击成功率、翻转样本数量、总样本数
    return success_rate, int(flip_count), int(total)


# ==========================================
# 第二部分：sample_data（抽样）
# ==========================================
def sample_data(images, labels, ratio=0.1):
    """
    从完整数据集中随机抽取 ratio 比例的数据
    images: 所有图片数据，numpy数组，形状为 [N, 32, 32, 3]
    labels: 所有标签，numpy数组，形状为 [N]
    ratio: 抽取比例，默认0.1（即10%）
    返回: 抽取后的图片和标签
    """
    num_samples = len(images)
    sample_count = int(num_samples * ratio)
    indices = np.random.choice(num_samples, sample_count, replace=False)
    sampled_images = images[indices]
    sampled_labels = labels[indices]
    return sampled_images, sampled_labels


# ==========================================
# 第三部分：compute_all_metrics（攻击鲁棒性指标）
# ==========================================
def compute_all_metrics(clean_acc, attack_acc, all_attack_accs=None):
    """
    计算所有评价指标（攻击鲁棒性）
    clean_acc: 干净样本上的准确率
    attack_acc: 攻击样本上的准确率
    all_attack_accs: 多个攻击强度下的准确率列表，用于算最差性能
    """
    retention = attack_acc / clean_acc if clean_acc > 0 else 0
    degradation = 1 - retention
    worst_case = min(all_attack_accs) if all_attack_accs else attack_acc

    return {
        "clean_accuracy": clean_acc,
        "attack_accuracy": attack_acc,
        "performance_retention": retention,
        "performance_degradation": degradation,
        "worst_case_accuracy": worst_case,
    }


# ==========================================
# 第四部分：compute_migration_metrics（迁移学习评估指标）
# ==========================================
def compute_migration_metrics(source_acc, target_acc, history_accs=None):
    """
    计算迁移学习评估指标
    source_acc: 源域准确率（迁移前）
    target_acc: 目标域准确率（迁移后）
    history_accs: 多次迁移的历史准确率列表，用于计算稳定性

    返回: 迁移保持率、稳定性、最差性能、迁移失败标记
    """
    # 迁移后有效性能保持率（比赛要求≥90%）迁移后性能是迁移前的百分之多少
    retention = target_acc / source_acc if source_acc > 0 else 0

    # 迁移稳定性：多次迁移的标准差（比赛要求波动±5%以内）
    stability = np.std(history_accs) if history_accs and len(history_accs) > 1 else 0

    # 最差迁移性能
    worst = min(history_accs) if history_accs else target_acc

    # 迁移失败检测：保持率<90% 或 性能骤降
    failed = retention < 0.9

    return {
        "migration_retention": retention,           # 迁移保持率（目标≥0.9）
        "migration_stability": stability,           # 稳定性（标准差，目标±0.05）
        "worst_migration_acc": worst,               # 最差迁移性能
        "migration_failed": failed,                 # 是否迁移失败
    }

# ==========================================
# 第五部分：batch_compute_attack_metrics（多个攻击强度下的鲁棒性指标）
# ==========================================
def batch_compute_attack_metrics(clean_acc, attack_results):
    """
    批量计算多个攻击强度下的鲁棒性指标

    Args:
        clean_acc: 干净样本准确率
        attack_results: list of dict, 每个dict包含 {'attack_name': str, 'attack_acc': float, 'params': dict}

    Returns:
        dict: 包含所有攻击强度的指标汇总
    """
    results = {}
    all_accs = []

    for result in attack_results:
        attack_name = result['attack_name']
        attack_acc = result['attack_acc']
        all_accs.append(attack_acc)

        metrics = compute_all_metrics(clean_acc, attack_acc, all_accs)
        results[attack_name] = {
            'attack_accuracy': attack_acc,
            'performance_retention': metrics['performance_retention'],
            'performance_degradation': metrics['performance_degradation'],
            'params': result.get('params', {})
        }

    # 添加汇总指标
    results['summary'] = {
        'worst_case_accuracy': min(all_accs),
        'avg_attack_accuracy': np.mean(all_accs),
        'num_attacks': len(attack_results)
    }

    return results

# ==========================================
# 第六部分：compute_comprehensive_robustness_metrics（全面的鲁棒性指标）
# ==========================================
def compute_comprehensive_robustness_metrics(clean_acc, attack_accs_by_type):
    """
    计算全面的鲁棒性指标

    Args:
        clean_acc: 干净样本准确率
        attack_accs_by_type: dict, 不同攻击类型和强度的准确率
            eg: {'FGSM': {'eps_0.01': 0.85, 'eps_0.03': 0.78}, ...}

    Returns:
        dict: 综合鲁棒性指标
    """
    results = {}

    for attack_type, accs in attack_accs_by_type.items():
        attack_acc_list = list(accs.values())
        all_attack_accs = attack_acc_list

        # 计算各强度指标
        for eps, acc in accs.items():
            metrics = compute_all_metrics(clean_acc, acc, all_attack_accs)
            results[f"{attack_type}_{eps}"] = metrics

    # 汇总所有攻击的最差情况
    all_accs = []
    for attack_type, accs in attack_accs_by_type.items():
        all_accs.extend(accs.values())

    results['global_summary'] = {
        'global_worst_case': min(all_accs),
        'global_avg_case': np.mean(all_accs),
        'max_degradation': max([1 - acc/clean_acc for acc in all_accs]),
        'num_attacks_tested': len(all_accs)
    }

    return results

# ==========================================
# 第七部分：format_metrics_to_table（将指标格式化为标准化表格）和 check_threshold（指标是否满足阈值要求）
# ==========================================
def check_threshold(category, metric_name, value):
    """
    检查指标是否满足阈值要求

    Args:
        category: 指标类别
        metric_name: 指标名称
        value: 指标值

    Returns:
        bool or None: True=达标, False=不达标, None=无阈值要求
    """
    thresholds = {
        'performance_retention': 0.9,      # 性能保持率 ≥ 90%
        'migration_retention': 0.9,        # 迁移保持率 ≥ 90%
        'migration_stability': 0.05,       # 迁移稳定性 ≤ 5%
        'worst_case_accuracy': 0.7,        # 最差性能 ≥ 70%
        'clean_accuracy': 0.85,            # 干净准确率 ≥ 85%
        'attack_accuracy': 0.7,            # 攻击准确率 ≥ 70%
        'performance_degradation': 0.3,    # 性能退化 ≤ 30%
    }

    if metric_name in thresholds:
        if metric_name in ['migration_stability', 'performance_degradation']:
            # 越小越好的指标
            return value <= thresholds[metric_name]
        else:
            # 越大越好的指标
            return value >= thresholds[metric_name]
    return None  # 无阈值要求

def format_metrics_to_table(metrics_dict, attack_type=None):
    """
    将指标格式化为标准化表格

    Args:
        metrics_dict: 指标字典
        attack_type: 攻击类型名称

    Returns:
        pd.DataFrame: 标准化的指标表格
    """
    rows = []
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    for key, value in metrics_dict.items():
        if isinstance(value, dict):
            # 递归处理嵌套字典
            for sub_key, sub_value in value.items():
                # 跳过summary中的嵌套字典
                if isinstance(sub_value, dict):
                    for sub_sub_key, sub_sub_value in sub_value.items():
                        rows.append({
                            'timestamp': timestamp,
                            'attack_type': attack_type or 'N/A',
                            'metric_category': f"{key}.{sub_key}",
                            'metric_name': sub_sub_key,
                            'metric_value': sub_sub_value,
                            'is_threshold_met': check_threshold(f"{key}.{sub_key}", sub_sub_key, sub_sub_value)
                        })
                else:
                    rows.append({
                        'timestamp': timestamp,
                        'attack_type': attack_type or 'N/A',
                        'metric_category': key,
                        'metric_name': sub_key,
                        'metric_value': sub_value,
                        'is_threshold_met': check_threshold(key, sub_key, sub_value)
                    })
        else:
            rows.append({
                'timestamp': timestamp,
                'attack_type': attack_type or 'N/A',
                'metric_category': 'base',
                'metric_name': key,
                'metric_value': value,
                'is_threshold_met': check_threshold('base', key, value)
            })

    return pd.DataFrame(rows)

# ==========================================
# 第八部分：Excel导出功能
# ==========================================
def flatten_metrics_dict(d, parent_key='', sep='_'):
    """
    扁平化嵌套字典，用于Excel导出

    Args:
        d: 嵌套字典
        parent_key: 父级键名
        sep: 分隔符

    Returns:
        dict: 扁平化后的字典
    """
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_metrics_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def calculate_summary_statistics(metrics_list):
    """
    计算汇总统计

    Args:
        metrics_list: 指标列表

    Returns:
        dict: 汇总统计
    """
    if not metrics_list:
        return {}
    df = pd.DataFrame(metrics_list)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()  # type: ignore
    summary = {}
    for col in numeric_cols:
        summary[f'{col}_mean'] = df[col].mean()
        summary[f'{col}_std'] = df[col].std()
        summary[f'{col}_min'] = df[col].min()
        summary[f'{col}_max'] = df[col].max()
    return summary

def check_all_thresholds(metrics_list):
    """
    检查所有指标是否满足阈值

    Args:
        metrics_list: 指标列表

    Returns:
        list: 阈值检查结果
    """
    results = []
    thresholds = METRICS_CONFIG['thresholds']

    for i, metrics in enumerate(metrics_list):
        for key, value in metrics.items():
            if key in thresholds:
                threshold = thresholds[key]
                # 判断是越小越好还是越大越好
                if key in ['migration_stability', 'performance_degradation']:
                    is_met = value <= threshold
                else:
                    is_met = value >= threshold
                results.append({
                    'test_id': i,
                    'metric': key,
                    'value': value,
                    'threshold': threshold,
                    'met': is_met,
                    'status': '✅ 达标' if is_met else '❌ 不达标'
                })
    return results

def export_metrics_to_excel(metrics_data, output_path=None, output_dir="results"):
    """
    导出指标台账到Excel

    Args:
        metrics_data: dict 或 list of dict
        output_path: 输出路径（如果为None则自动生成）
        output_dir: 输出目录

    Returns:
        str: 生成的文件路径
    """
    # 创建输出目录
    os.makedirs(output_dir, exist_ok=True)

    # 生成输出路径
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = os.path.join(output_dir, f"metrics_ledger_{timestamp}.xlsx")

    # 确保是列表格式
    if isinstance(metrics_data, dict):
        metrics_list = [flatten_metrics_dict(metrics_data)]
    elif isinstance(metrics_data, list):
        metrics_list = metrics_data
    else:
        raise ValueError("metrics_data 必须是 dict 或 list")

    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        # 1. 写入主指标表
        df_main = pd.DataFrame(metrics_list)
        df_main.to_excel(writer, sheet_name='Main_Metrics', index=False)

        # 2. 写入汇总统计
        summary = calculate_summary_statistics(metrics_list)
        if summary:
            pd.DataFrame([summary]).to_excel(writer, sheet_name='Summary', index=False)

        # 3. 写入阈值检查
        threshold_check = check_all_thresholds(metrics_list)
        if threshold_check:
            df_threshold = pd.DataFrame(threshold_check)
            df_threshold.to_excel(writer, sheet_name='Threshold_Check', index=False)

        # 4. 写入合规性总结
        compliance_summary = generate_compliance_summary(threshold_check)
        pd.DataFrame([compliance_summary]).to_excel(writer, sheet_name='Compliance', index=False)

    print(f"✅ Excel台账已导出: {output_path}")
    return output_path

def generate_compliance_summary(threshold_check):
    """
    生成合规性总结

    Args:
        threshold_check: check_all_thresholds 的结果

    Returns:
        dict: 合规性总结
    """
    if not threshold_check:
        return {'total_metrics': 0, 'passed': 0, 'failed': 0, 'all_passed': False}

    total = len(threshold_check)
    passed = sum(1 for item in threshold_check if item['met'])
    failed = total - passed

    return {
        'total_metrics': total,
        'passed': passed,
        'failed': failed,
        'pass_rate': passed / total if total > 0 else 0,
        'all_passed': failed == 0,
        'check_time': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

# ==========================================
# 第九部分：完整台账生成（整合所有功能）
# ==========================================
def generate_complete_ledger(
    clean_acc,
    attack_results,
    source_acc=None,
    target_acc=None,
    inference_count=None,
    data_ratio=None,
    elapsed_time=None,
    output_dir="results"
):
    """
    生成完整的测试台账（一键生成所有指标和报告）

    Args:
        clean_acc: 干净样本准确率
        attack_results: 攻击结果列表
        source_acc: 源域准确率（迁移前）
        target_acc: 目标域准确率（迁移后）
        inference_count: 推理次数
        data_ratio: 数据使用比例
        elapsed_time: 测试耗时
        output_dir: 输出目录

    Returns:
        dict: 完整的台账数据
    """
    os.makedirs(output_dir, exist_ok=True)

    # 1. 计算攻击指标
    attack_metrics = batch_compute_attack_metrics(clean_acc, attack_results)

    # 2. 计算迁移指标（如果有）
    migration_metrics = None
    if source_acc is not None and target_acc is not None:
        migration_metrics = compute_migration_metrics(source_acc, target_acc)

    # 3. 构建完整台账
    ledger = {
        'test_info': {
            'timestamp': datetime.now().isoformat(),
            'inference_count': inference_count,
            'data_ratio': data_ratio,
            'elapsed_time': elapsed_time
        },
        'migration_metrics': migration_metrics,
        'attack_metrics': attack_metrics,
        'compliance': {
            'inference_compliant': inference_count <= 1000 if inference_count else None,
            'data_compliant': data_ratio <= 0.10 if data_ratio else None,
            'time_compliant': elapsed_time <= 300 if elapsed_time else None
        }
    }

    # 4. 导出Excel
    excel_path = export_metrics_to_excel(ledger, output_dir=output_dir)

    # 5. 导出JSON
    json_path = os.path.join(output_dir, f"ledger_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    with open(json_path, 'w') as f:
        json.dump(ledger, f, indent=2)

    print(f"✅ 完整台账已生成:")
    print(f"   Excel: {excel_path}")
    print(f"   JSON: {json_path}")

    return ledger

# ==========================================
# 第十部分：线程安全的指标收集器
# ==========================================
class MetricsCollector:
    """
    线程安全的指标收集器，用于并行测试

    使用示例:
        collector = MetricsCollector()

        # 在多个线程/进程中
        collector.add_metrics('test_001', {
            'clean_accuracy': 0.92,
            'attack_accuracy': 0.78,
            'performance_retention': 0.85
        })

        # 获取所有结果
        all_metrics = collector.get_all_metrics()

        # 获取汇总统计
        summary = collector.get_summary()
    """

    def __init__(self):
        self._lock = threading.Lock()
        self.metrics = defaultdict(list)
        self._start_time = datetime.now()

    def add_metrics(self, test_id, metrics_dict):
        with self._lock:
            self.metrics[test_id].append({
                'timestamp': datetime.now().isoformat(),
                'test_id': str(test_id),
                **metrics_dict
            })

    def add_attack_result(self, test_id, attack_name, attack_acc, clean_acc, params=None):
        metrics = compute_all_metrics(clean_acc, attack_acc, [attack_acc])
        metrics.update({
            'attack_name': attack_name,
            'attack_accuracy': attack_acc,
            'clean_accuracy': clean_acc,
            'params': str(params) if params else '{}'  # 转为字符串
        })
        self.add_metrics(test_id, metrics)

    def get_all_metrics(self):
        with self._lock:
            return {k: v.copy() for k, v in self.metrics.items()}

    def get_flat_metrics(self):
        all_metrics = self.get_all_metrics()
        flat_list = []
        for test_id, metrics_list in all_metrics.items():
            for metrics in metrics_list:
                flat_list.append(metrics)
        return flat_list

    def get_summary(self):
        flat_list = self.get_flat_metrics()
        if not flat_list:
            return {'total_tests': 0, 'total_metrics': 0, 'status': 'no_data'}

        df = pd.DataFrame(flat_list)
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        summary = {
            'total_tests': len(df['test_id'].unique()),
            'total_metrics': len(df),
            'start_time': self._start_time.isoformat(),
            'end_time': datetime.now().isoformat()
        }

        for col in numeric_cols:
            if col not in ['test_id']:
                summary[f'{col}_mean'] = float(df[col].mean())
                summary[f'{col}_std'] = float(df[col].std())
                summary[f'{col}_min'] = float(df[col].min())
                summary[f'{col}_max'] = float(df[col].max())

        return summary

    def export_to_excel(self, output_path=None, output_dir="results"):
        """导出所有收集的指标到Excel（修复版）"""
        import os
        os.makedirs(output_dir, exist_ok=True)

        if output_path is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = os.path.join(output_dir, f"collector_metrics_{timestamp}.xlsx")

        flat_list = self.get_flat_metrics()
        if not flat_list:
            print("⚠️ 没有收集到任何指标")
            return None

        df = pd.DataFrame(flat_list)

        with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
            # 1. 写入所有指标
            df.to_excel(writer, sheet_name='All_Metrics', index=False)

            # 2. 写入汇总统计
            summary = self.get_summary()
            pd.DataFrame([summary]).to_excel(writer, sheet_name='Summary', index=False)

            # 3. 按test_id分组统计（只选数值列）
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if 'test_id' in numeric_cols:
                numeric_cols.remove('test_id')

            if numeric_cols:
                # 只对数值列进行聚合
                grouped = df.groupby('test_id')[numeric_cols].agg(['mean', 'std', 'min', 'max'])  # type: ignore
                grouped.to_excel(writer, sheet_name='Grouped_Stats')
            else:
                # 记录数统计
                count_stats = df.groupby('test_id').size().reset_index(name='record_count')
                count_stats.to_excel(writer, sheet_name='Record_Counts', index=False)

            # 4. 添加一个Sheet显示数据概览
            overview = pd.DataFrame({
                '总测试数': [len(df['test_id'].unique())],
                '总记录数': [len(df)],
                '数值列': [', '.join(numeric_cols) if numeric_cols else '无'],
                '导出时间': [datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
            })
            overview.to_excel(writer, sheet_name='Overview', index=False)

        print(f"✅ 收集器数据已导出: {output_path}")
        return output_path

    def clear(self):
        with self._lock:
            self.metrics.clear()
            self._start_time = datetime.now()

    def get_test_count(self):
        with self._lock:
            return len(self.metrics)

    def get_metric_count(self):
        with self._lock:
            return sum(len(v) for v in self.metrics.values())


# ==========================================
# 第十一部分：并行测试辅助函数
# ==========================================
def run_parallel_tests(test_configs, collector, num_workers=4):
    """
    并行运行多个测试（需要配合 multiprocessing 使用）

    Args:
        test_configs: list of dict, 每个配置包含测试参数
        collector: MetricsCollector 实例
        num_workers: 并行工作进程数

    Returns:
        dict: 测试结果汇总
    """
    from multiprocessing import Pool
    import time

    def run_single_test(config):
        """单个测试任务"""
        test_id = config.get('test_id', f"test_{time.time()}")
        clean_acc = config.get('clean_acc', 0.92)
        attack_results = config.get('attack_results', [])

        # 计算攻击指标
        batch_metrics = batch_compute_attack_metrics(clean_acc, attack_results)

        # 添加到收集器
        collector.add_metrics(test_id, {
            'clean_accuracy': clean_acc,
            'batch_metrics': batch_metrics,
            'config': config
        })

        return test_id

    # 使用进程池并行执行
    with Pool(processes=num_workers) as pool:
        results = pool.map(run_single_test, test_configs)

    return {
        'total_tests': len(results),
        'completed_tests': results,
        'summary': collector.get_summary()
    }

# ==========================================
# 完整测试流程（带计时）
# ==========================================
@timing_decorator
def run_full_test_pipeline(clean_acc, attack_results, source_acc=None, target_acc=None):
    """
    运行完整的测试流程（带计时）

    Returns:
        dict: 包含所有指标和计时信息
    """
    print("🚀 开始完整测试流程...")

    # 1. 攻击指标计算
    with timing_context("attack_metrics"):
        attack_metrics = batch_compute_attack_metrics(clean_acc, attack_results)

    # 2. 迁移指标计算（如果有）
    migration_metrics = None
    if source_acc is not None and target_acc is not None:
        with timing_context("migration_metrics"):
            migration_metrics = compute_migration_metrics(source_acc, target_acc)

    # 3. 汇总结果
    result = {
        'attack_metrics': attack_metrics,
        'migration_metrics': migration_metrics,
        'timings': get_all_timings(),
        'total_time': sum(get_all_timings().values())
    }

    # 打印计时汇总
    print_timing_summary()

    return result

# ==========================================
# 第六部分：用假数据测试所有函数（修复打印逻辑）
# ==========================================
if __name__ == "__main__":
    print("=" * 60)
    print("🧪 eval_tool.py 完整功能测试")
    print("=" * 60)

    # ==========================================
    # 测试1: compute_accuracy
    # ==========================================
    print("\n[测试1] compute_accuracy - 攻击成功率计算")
    fake_preds = [0, 1, 1, 0, 0]
    fake_labels = [0, 1, 0, 0, 0]
    success_rate, flip_count, total = compute_accuracy(fake_preds, fake_labels)
    print(f"  ✅ 攻击成功率: {success_rate:.2%}，翻转样本数：{flip_count}，总样本：{total}")
    assert success_rate == 0.2, "攻击成功率计算错误"
    print("  ✅ 测试通过")

    # ==========================================
    # 测试2: sample_data
    # ==========================================
    print("\n[测试2] sample_data - 数据采样功能")
    fake_images = np.random.randn(100, 32, 32, 3)
    fake_labels = np.random.randint(0, 10, size=100)
    sampled_imgs, sampled_lbls = sample_data(fake_images, fake_labels, 0.1)
    print(f"  ✅ 原始: 100张 → 采样后: {len(sampled_imgs)}张 (期望: 10张)")
    assert len(sampled_imgs) == 10, "数据采样数量错误"
    print("  ✅ 测试通过")

    # ==========================================
    # 测试3: compute_all_metrics
    # ==========================================
    print("\n[测试3] compute_all_metrics - 攻击鲁棒性指标")
    clean_acc = 0.92
    attack_acc = 0.78
    all_accs = [0.78, 0.75, 0.72]
    metrics = compute_all_metrics(clean_acc, attack_acc, all_accs)
    print(f"  ✅ 性能保持率: {metrics['performance_retention']:.2%} (目标≥90%)")
    print(f"  ✅ 退化幅度: {metrics['performance_degradation']:.2%}")
    print(f"  ✅ 最差性能: {metrics['worst_case_accuracy']:.2%}")
    assert 0 <= metrics['performance_retention'] <= 1, "性能保持率应在0-1之间"
    print("  ✅ 测试通过")

    # ==========================================
    # 测试4: compute_migration_metrics
    # ==========================================
    print("\n[测试4] compute_migration_metrics - 迁移学习指标")
    source_acc = 0.95
    target_acc = 0.88
    history = [0.90, 0.88, 0.92, 0.87]
    mig = compute_migration_metrics(source_acc, target_acc, history)
    print(f"  ✅ 迁移保持率: {mig['migration_retention']:.2%} {'✅达标' if mig['migration_retention'] >= 0.9 else '❌不达标'}")
    print(f"  ✅ 迁移稳定性: {mig['migration_stability']:.4f} (标准差，目标≤0.05)")
    print(f"  ✅ 最差迁移性能: {mig['worst_migration_acc']:.2%}")
    print(f"  ✅ 迁移失败标记: {mig['migration_failed']}")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试5: 迁移失败场景
    # ==========================================
    print("\n[测试5] compute_migration_metrics - 迁移失败检测")
    bad_target = 0.80
    mig_bad = compute_migration_metrics(source_acc, bad_target)
    print(f"  ✅ 保持率: {mig_bad['migration_retention']:.2%} → 迁移失败: {mig_bad['migration_failed']}")
    assert mig_bad['migration_failed'] == True, "应该检测到迁移失败"
    print("  ✅ 测试通过")

    # ==========================================
    # 测试6: batch_compute_attack_metrics
    # ==========================================
    print("\n[测试6] batch_compute_attack_metrics - 批量攻击指标")
    attack_results = [
        {'attack_name': 'FGSM_eps001', 'attack_acc': 0.85, 'params': {'eps': 0.01}},
        {'attack_name': 'FGSM_eps003', 'attack_acc': 0.78, 'params': {'eps': 0.03}},
        {'attack_name': 'FGSM_eps005', 'attack_acc': 0.72, 'params': {'eps': 0.05}}
    ]
    clean_acc = 0.92
    batch_results = batch_compute_attack_metrics(clean_acc, attack_results)
    print(f"  ✅ 最差性能: {batch_results['summary']['worst_case_accuracy']:.2%}")
    print(f"  ✅ 平均攻击准确率: {batch_results['summary']['avg_attack_accuracy']:.2%}")
    print(f"  ✅ 攻击数量: {batch_results['summary']['num_attacks']}")
    for name, metrics in batch_results.items():
        if name != 'summary':
            print(f"    - {name}: 保持率={metrics['performance_retention']:.2%}")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试7: compute_comprehensive_robustness_metrics
    # ==========================================
    print("\n[测试7] compute_comprehensive_robustness_metrics - 综合鲁棒性")
    attack_accs_by_type = {
        'FGSM': {
            'eps_0.01': 0.85,
            'eps_0.03': 0.78,
            'eps_0.05': 0.72
        },
        'PGD': {
            'eps_0.01': 0.83,
            'eps_0.03': 0.75,
            'eps_0.05': 0.68
        },
        'BIM': {
            'eps_0.01': 0.84,
            'eps_0.03': 0.76,
            'eps_0.05': 0.70
        }
    }
    comprehensive_results = compute_comprehensive_robustness_metrics(clean_acc, attack_accs_by_type)
    print(f"  ✅ 全局最差性能: {comprehensive_results['global_summary']['global_worst_case']:.2%}")
    print(f"  ✅ 全局平均性能: {comprehensive_results['global_summary']['global_avg_case']:.2%}")
    print(f"  ✅ 最大性能退化: {comprehensive_results['global_summary']['max_degradation']:.2%}")
    print(f"  ✅ 测试攻击数量: {comprehensive_results['global_summary']['num_attacks_tested']}")
    for i, (attack_name, metrics) in enumerate(comprehensive_results.items()):
        if attack_name != 'global_summary' and i < 3:
            print(f"    - {attack_name}: 保持率={metrics['performance_retention']:.2%}")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试8: format_metrics_to_table
    # ==========================================
    print("\n[测试8] format_metrics_to_table - 表格格式化")
    df = format_metrics_to_table(batch_results, 'FGSM')
    print(f"  ✅ 表格形状: {df.shape[0]}行 x {df.shape[1]}列")
    print(f"  ✅ 列名: {list(df.columns)}")
    print(f"  ✅ 阈值检查列存在: {'is_threshold_met' in df.columns}")
    print("\n  表格预览（前3行）:")
    print(df.head(3).to_string(index=False))
    print("  ✅ 测试通过")

    # ==========================================
    # 测试9: check_threshold
    # ==========================================
    print("\n[测试9] check_threshold - 阈值检查")
    test_cases = [
        ('performance_retention', 0.92, True),
        ('performance_retention', 0.85, False),
        ('migration_stability', 0.03, True),
        ('migration_stability', 0.08, False),
        ('unknown_metric', 0.5, None)
    ]
    for metric_name, value, expected in test_cases:
        result = check_threshold('test', metric_name, value)
        status = '✅' if result == expected else '❌'
        print(f"  {status} {metric_name}={value} → {result} (期望: {expected})")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试10: export_metrics_to_excel
    # ==========================================
    print("\n[测试10] export_metrics_to_excel - Excel导出")
    import os
    os.makedirs("test_results", exist_ok=True)

    excel_path1 = export_metrics_to_excel(
        batch_results,
        output_path="test_results/batch_metrics.xlsx"
    )
    print(f"  ✅ 批量指标Excel: {excel_path1}")
    assert os.path.exists(excel_path1), "Excel文件未生成"

    multi_results = [
        {'performance_retention': 0.92, 'clean_accuracy': 0.95},
        {'performance_retention': 0.88, 'clean_accuracy': 0.93},
        {'performance_retention': 0.90, 'clean_accuracy': 0.94}
    ]
    excel_path2 = export_metrics_to_excel(
        multi_results,
        output_path="test_results/multi_metrics.xlsx"
    )
    print(f"  ✅ 多轮指标Excel: {excel_path2}")
    assert os.path.exists(excel_path2), "Excel文件未生成"

    import openpyxl
    wb = openpyxl.load_workbook(excel_path1)
    print(f"  ✅ Excel包含的sheet: {wb.sheetnames}")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试11: generate_complete_ledger
    # ==========================================
    print("\n[测试11] generate_complete_ledger - 完整台账生成")
    try:
        # 检查函数是否存在
        if 'generate_complete_ledger' in dir():
            ledger = generate_complete_ledger(
                clean_acc=0.92,
                attack_results=attack_results,
                source_acc=0.95,
                target_acc=0.88,
                inference_count=960,
                data_ratio=0.048,
                elapsed_time=63.69,
                output_dir="test_results"
            )
            print(f"  ✅ 完整台账已生成")
            print(f"  ✅ 推理次数合规: {ledger['compliance']['inference_compliant']}")
            print(f"  ✅ 数据依赖合规: {ledger['compliance']['data_compliant']}")
            print(f"  ✅ 时间合规: {ledger['compliance']['time_compliant']}")
        else:
            print("  ⚠️ generate_complete_ledger 未实现，跳过测试")
    except NameError:
        print("  ⚠️ generate_complete_ledger 未实现，跳过测试")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试12: 性能基准测试
    # ==========================================
    print("\n[测试12] 性能基准测试")
    import time

    start = time.time()
    for _ in range(1000):
        compute_all_metrics(0.92, 0.78, [0.78, 0.75, 0.72])
    elapsed = time.time() - start
    print(f"  ✅ compute_all_metrics 1000次调用: {elapsed:.3f}秒")

    start = time.time()
    for _ in range(100):
        batch_compute_attack_metrics(0.92, attack_results)
    elapsed = time.time() - start
    print(f"  ✅ batch_compute_attack_metrics 100次调用: {elapsed:.3f}秒")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试13: MetricsCollector 并行收集器
    # ==========================================
    print("\n[测试13] MetricsCollector - 线程安全指标收集器")

    # 创建收集器
    collector = MetricsCollector()

    # 模拟多线程添加数据
    test_data = [
        ('test_001', 0.92, [0.85, 0.78, 0.72]),
        ('test_002', 0.91, [0.84, 0.77, 0.70]),
        ('test_003', 0.93, [0.86, 0.79, 0.73])
    ]

    for test_id, clean_acc_val, attack_accs in test_data:
        for i, attack_acc_val in enumerate(attack_accs):
            collector.add_attack_result(
                test_id=test_id,
                attack_name=f"FGSM_eps_{i+1:02d}",
                attack_acc=attack_acc_val,
                clean_acc=clean_acc_val,
                params={'eps': (i+1) * 0.02}
            )

    print(f"  ✅ 添加了 {collector.get_test_count()} 个测试")
    print(f"  ✅ 总共 {collector.get_metric_count()} 条指标")

    # 获取所有指标
    all_metrics = collector.get_all_metrics()
    for test_id, metrics_list in all_metrics.items():
        print(f"  📊 {test_id}: {len(metrics_list)} 条记录")

    # 汇总统计
    summary = collector.get_summary()
    print(f"  ✅ 总测试数: {summary['total_tests']}")
    print(f"  ✅ 总指标数: {summary['total_metrics']}")
    print(f"  ✅ 平均性能保持率: {summary.get('performance_retention_mean', 'N/A'):.3f}")

    # 导出Excel
    excel_path = collector.export_to_excel(output_dir="test_results")
    print(f"  ✅ 收集器数据已导出")
    print("  ✅ 测试通过")

    # ==========================================
    # 测试14: 性能监控装饰器
    # ==========================================
    print("\n[测试14] 性能监控装饰器")

    # 1. 手动计时 compute_all_metrics
    start = time.time()
    result = compute_all_metrics(clean_acc, 0.78, [0.78, 0.75, 0.72])
    elapsed1 = time.time() - start
    print(f"  ✅ compute_all_metrics 耗时: {elapsed1:.4f}秒")

    # 2. 手动计时 batch_compute_attack_metrics
    start = time.time()
    batch_result = batch_compute_attack_metrics(clean_acc, attack_results)
    elapsed2 = time.time() - start
    print(f"  ✅ batch_compute_attack_metrics 耗时: {elapsed2:.4f}秒")

    # 3. 上下文管理器计时（如果有）
    try:
        with timing_context("模拟处理"):
            time.sleep(0.05)
            _ = [i**2 for i in range(1000)]
        print(f"  ✅ 上下文计时完成")
    except NameError:
        print("  ⚠️ timing_context 未实现，跳过")
    except Exception as e:
        print(f"  ⚠️ 上下文计时失败: {e}")

    # 4. 完整流程计时（如果有）
    try:
        if 'run_full_test_pipeline' in dir():
            start = time.time()
            full_result = run_full_test_pipeline(
                clean_acc=0.92,
                attack_results=attack_results,
                source_acc=0.95,
                target_acc=0.88
            )
            elapsed3 = time.time() - start
            print(f"  ✅ 完整流程测试完成，耗时: {elapsed3:.4f}秒")
        else:
            print("  ⚠️ run_full_test_pipeline 未实现，跳过")
    except NameError:
        print("  ⚠️ run_full_test_pipeline 未实现，跳过")

    print("\n" + "=" * 60)
    print("⏱️ 性能计时汇总")
    print("=" * 60)
    print(f"  compute_all_metrics: {elapsed1:.4f}秒")
    print(f"  batch_compute_attack_metrics: {elapsed2:.4f}秒")
    total_time = elapsed1 + elapsed2
    print("-" * 60)
    print(f"  总耗时: {total_time:.4f}秒")
    if total_time <= 300:
        print(f"  ✅ 符合 ≤300秒 要求")
    else:
        print(f"  ❌ 超出限制 {(total_time - 300):.2f}秒")
    print("=" * 60)

    print("  ✅ 测试通过")

    # ==========================================
    # 总结
    # ==========================================
    print("\n" + "=" * 60)
    print("🎉 所有测试通过！eval_tool.py 功能完整 ✅")
    print("=" * 60)
    print("\n📊 测试覆盖的功能模块:")
    print("  ✅ 攻击成功率计算 (compute_accuracy)")
    print("  ✅ 数据采样 (sample_data)")
    print("  ✅ 攻击鲁棒性指标 (compute_all_metrics)")
    print("  ✅ 迁移学习指标 (compute_migration_metrics)")
    print("  ✅ 批量攻击指标 (batch_compute_attack_metrics)")
    print("  ✅ 综合鲁棒性指标 (compute_comprehensive_robustness_metrics)")
    print("  ✅ 表格格式化 (format_metrics_to_table)")
    print("  ✅ 阈值检查 (check_threshold)")
    print("  ✅ Excel导出 (export_metrics_to_excel)")
    print("  ✅ 完整台账生成 (generate_complete_ledger)")
    print("  ✅ 线程安全收集器 (MetricsCollector)")
    print("  ✅ 性能监控装饰器 (timing_decorator)")
    print("  ✅ 性能基准测试")
    print("=" * 60)

    # 显示生成的文件
    print("\n📁 生成的测试文件:")
    import glob
    test_files = glob.glob("test_results/*")
    if test_files:
        for f in test_files:
            size = os.path.getsize(f) / 1024
            print(f"  📄 {os.path.basename(f)} ({size:.1f} KB)")
    else:
        print("  ⚠️ 没有生成测试文件")

🧪 eval_tool.py 完整功能测试

[测试1] compute_accuracy - 攻击成功率计算
  ✅ 攻击成功率: 20.00%，翻转样本数：1，总样本：5
  ✅ 测试通过

[测试2] sample_data - 数据采样功能
  ✅ 原始: 100张 → 采样后: 10张 (期望: 10张)
  ✅ 测试通过

[测试3] compute_all_metrics - 攻击鲁棒性指标
  ✅ 性能保持率: 84.78% (目标≥90%)
  ✅ 退化幅度: 15.22%
  ✅ 最差性能: 72.00%
  ✅ 测试通过

[测试4] compute_migration_metrics - 迁移学习指标
  ✅ 迁移保持率: 92.63% ✅达标
  ✅ 迁移稳定性: 0.0192 (标准差，目标≤0.05)
  ✅ 最差迁移性能: 87.00%
  ✅ 迁移失败标记: False
  ✅ 测试通过

[测试5] compute_migration_metrics - 迁移失败检测
  ✅ 保持率: 84.21% → 迁移失败: True
  ✅ 测试通过

[测试6] batch_compute_attack_metrics - 批量攻击指标
  ✅ 最差性能: 72.00%
  ✅ 平均攻击准确率: 78.33%
  ✅ 攻击数量: 3
    - FGSM_eps001: 保持率=92.39%
    - FGSM_eps003: 保持率=84.78%
    - FGSM_eps005: 保持率=78.26%
  ✅ 测试通过

[测试7] compute_comprehensive_robustness_metrics - 综合鲁棒性
  ✅ 全局最差性能: 68.00%
  ✅ 全局平均性能: 76.78%
  ✅ 最大性能退化: 26.09%
  ✅ 测试攻击数量: 9
    - FGSM_eps_0.01: 保持率=92.39%
    - FGSM_eps_0.03: 保持率=84.78%
    - FGSM_eps_0.05: 保持率=78.26%
  ✅ 测试通过

[测试8] format_metrics_to_table - 表格格式化
  ✅ 表格形状: 15行 x 6列
  ✅ 列名: ['timestamp'